# M14 · Encoders & contrastive training

_Curriculum · Domain 2 · Retrieval & Representation_

**Fine-tune text encoders so positives pull together and confusing negatives push apart.**

We simulate encoder embeddings with small numpy vectors. The InfoNCE loss is $\ell=-\log\frac{\exp(s_+/\tau)}{\sum_j\exp(s_j/\tau)}$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(14)

## Simulated encoder outputs

No transformer weights are downloaded. These vectors stand in for brief and creator embeddings after a text encoder.

In [ ]:
query = np.array([1.0, 0.2, 0.1, 0.0])
positive = np.array([0.9, 0.25, 0.1, 0.05])
easy_negative = np.array([0.0, 0.1, 0.9, 0.2])
hard_negative = np.array([0.75, 0.30, 0.15, 0.10])
items = np.vstack([positive, easy_negative, hard_negative])
labels = np.array(["positive", "easy negative", "hard negative"])

def normalize(x):
    return x / np.linalg.norm(x, axis=-1, keepdims=True)

query_n = normalize(query)
items_n = normalize(items)
cosines = items_n @ query_n

print(pd.DataFrame({"item": labels, "cosine": cosines}).round(3))

## Step 1 - Compute InfoNCE

A hard negative has a cosine close to the positive, so it steals probability mass.

In [ ]:
def info_nce(similarities, tau):
    logits = similarities / tau
    logits = logits - logits.max()
    weights = np.exp(logits)
    probs = weights / weights.sum()
    return -np.log(probs[0]), probs

tau = 0.1
loss_easy, probs_easy = info_nce(cosines[:2], tau)
loss_hard, probs_hard = info_nce(cosines, tau)

print("easy-only loss:", round(loss_easy, 4))
print("with-hard loss:", round(loss_hard, 4))

assert loss_hard > loss_easy

## Step 2 - Inspect the probabilities

The hard negative is useful because the model still assigns it meaningful probability.

In [ ]:
prob_table = pd.DataFrame({
    "item": labels,
    "probability_with_hard": probs_hard,
})

print(prob_table.round(4))

assert probs_hard[2] > probs_easy[1]

## Step 3 - Temperature changes sharpness

Lower temperature magnifies score gaps. Higher temperature spreads probability more evenly.

In [ ]:
taus = np.array([0.05, 0.10, 0.20, 0.50])
rows = []
for value in taus:
    loss_value, prob_value = info_nce(cosines, value)
    rows.append({"tau": value, "loss": loss_value, "positive_probability": prob_value[0]})

temp_df = pd.DataFrame(rows)

print(temp_df.round(4))

assert temp_df["loss"].min() >= 0

## Visualize the hard-negative effect

The loss jumps when the negative is close enough to be plausible.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(["easy only", "with hard"], [loss_easy, loss_hard], color=["#4c78a8", "#f58518"])
ax.set_ylabel("InfoNCE loss")
ax.set_title("hard negatives increase signal")
plt.show()

## Practice

Move `hard_negative` closer to `positive`, or raise `tau`. Re-run the notebook and explain how the loss and probabilities change.

In [ ]:
# Your turn:
